# NB14c — Secondary-Filter Validation (Option A Lock)

**Datum:** 2026-05-27
**Lock-Basis:** [ANN-012 V1 Tier-Architektur — Premium Core + Secondary Filters](../docs/decisions/ANN-012-v1-tier-architecture-premium-core-plus-filters.md)
**Vorgänger:** NB14 (5m-TF gelocked, PF 2.39 Hold-Out), NB14b (Probability-Cutoffs widerlegt)

**Ziel:** Finale Calibration der 3 V1-Profile via Filter-Stack auf Premium-Tier. Alle Profile teilen denselben Premium-Cutoff (`0.4096`) — sie differenzieren ausschließlich via Sekundär-Filter.

| Profil | Filter-Stack | Erwartung |
|---|---|---:|
| Aggressive | Premium pur | ~3.5 Sigs/Tag/Symbol |
| Balanced | Premium + HTF-Confirm (`htf_ltf_agree_bull`) | ~3.0 |
| Conservative | Premium + HTF-Confirm + NY-Session | ~1.5 |

**Quality-Anchor (ANN-010) muss jedes Profil bestehen:** PF ≥ 1.5 strict / MDD < 18% / CV < 0.25 / min Trades/Jahr ≥ 30.

**Output:** `/results/nb14c/profile_calibration_{date}.json` + per-profile CSVs.

**Erwartete Laufzeit:** ~10–15 Min Colab (re-use NB14-Trainings-Pattern, kein neues Modell-Training nötig falls Modell-Cache greift; sonst ~30s extra).

## Section 0 — Setup + Drive Mount + Module Sync

In [ ]:
import os, sys, subprocess, gc, json, importlib
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    subprocess.run(['rm', '-rf', '/tmp/pace-algo'])
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ecoNC/pace-algo.git',
                    '/tmp/pace-algo'], check=True)
    subprocess.run(f'mkdir -p {PROJECT_ROOT}/core/eval {PROJECT_ROOT}/core/analysis {PROJECT_ROOT}/core/router',
                   shell=True)
    subprocess.run(f'cp -rf /tmp/pace-algo/core/. {PROJECT_ROOT}/core/', shell=True)
    subprocess.run(f"find {PROJECT_ROOT}/core -type d -name __pycache__ -exec rm -rf {{}} +", shell=True)
    print('Core synced.')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
importlib.invalidate_caches()

RANDOM_SEED = 42
import numpy as np
np.random.seed(RANDOM_SEED)

RUN_DATE      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
try:
    GIT_COMMIT = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', '--short', 'HEAD'],
                                          text=True).strip()
except Exception:
    GIT_COMMIT = 'unknown'
EXPERIMENT_ID = f'nb14c_{RUN_TIMESTAMP}_{GIT_COMMIT}'
print(f'EXPERIMENT_ID: {EXPERIMENT_ID}')

In [ ]:
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'lightgbm>=4.3', 'pyarrow>=15.0'], capture_output=True)

import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

from core import config as cfg
from core.config import (
    FX_TRAIN_SYMBOLS, FX_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END,
)
from core.train.dataset import walk_forward_split, binary_label_for_long, NON_FEATURE_COLS
from core.train.lgbm_trainer import train_lgbm, trading_metrics_from_predictions
from core.analysis.product_metrics import signals_per_day, TF_BARS_PER_DAY
from core.analysis.quality_check import check_quality_anchor

# Konfiguration — V1 gelocked auf 5m, FX-Modell
TF = '5m'
R_VALUE = 1.5
WIN_R, LOSS_R = R_VALUE, 1.0
ALL_SYMBOLS = FX_TRAIN_SYMBOLS + FX_HOLDOUT_SYMBOLS

# Premium-Cutoff aus NB14/NB14b — gelocked
PREMIUM_CUTOFF = 0.4096   # ANN-012 §3 — der einzige sauber separierbare Tier

# NY-Session-Definition (UTC) — siehe ANN-012 §5 + R-13 Beleg
NY_SESSION_HOUR_START = 13   # 13:00 UTC = 9:00 EST (NY-Open)
NY_SESSION_HOUR_END   = 22   # 22:00 UTC = 18:00 EST (NY-Close)

# Filter-Definitionen für die 3 Profile (ANN-012)
PROFILES = {
    'Aggressive':   {'require_htf': False, 'require_ny': False},
    'Balanced':     {'require_htf': True,  'require_ny': False},
    'Conservative': {'require_htf': True,  'require_ny': True},
}

OUTPUT_DIR = Path(PROJECT_ROOT) / 'results' / 'nb14c'
for sub in ('metrics', 'summaries', 'config_snapshots'):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'TF: {TF}  |  Premium-Cutoff (locked): {PREMIUM_CUTOFF}')
print(f'NY-Session (UTC): {NY_SESSION_HOUR_START}:00–{NY_SESSION_HOUR_END}:00')
print(f'Train symbols: {FX_TRAIN_SYMBOLS}')
print(f'Hold-Out symbols: {FX_HOLDOUT_SYMBOLS}')
print(f'Output: {OUTPUT_DIR}')

# Config snapshot
snapshot_path = OUTPUT_DIR / 'config_snapshots' / f'{EXPERIMENT_ID}_config.json'
with open(snapshot_path, 'w') as f:
    json.dump({
        'experiment_id':   EXPERIMENT_ID,
        'run_date':        RUN_DATE,
        'git_commit':      GIT_COMMIT,
        'random_seed':     RANDOM_SEED,
        'tf':              TF,
        'premium_cutoff':  PREMIUM_CUTOFF,
        'ny_session_utc':  [NY_SESSION_HOUR_START, NY_SESSION_HOUR_END],
        'profiles':        PROFILES,
        'train_symbols':   FX_TRAIN_SYMBOLS,
        'holdout_symbols': FX_HOLDOUT_SYMBOLS,
    }, f, indent=2)
print(f'Config snapshot → {snapshot_path}')

## Section 1 — Load Extended Features + Train V1-FX-Modell

Identisch zu NB14/NB14b (gleiche Hyperparams, gleiches Seed) damit Predictions deterministisch reproduzierbar sind.

In [ ]:
if IN_COLAB:
    DATA_EXT = Path('/content/processed_v2')
    DATA_PROCESSED_LOCAL = Path('/content/processed')
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    DRIVE_BACKUP_PROCESSED = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    DATA_EXT.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED_LOCAL.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED_LOCAL.glob('labels_*.parquet'))) == 0 and DRIVE_BACKUP_PROCESSED.exists():
        subprocess.run(['rsync', '-ah', f'{DRIVE_BACKUP_PROCESSED}/', f'{DATA_PROCESSED_LOCAL}/'])
    if len(list(DATA_EXT.glob('*_extended.parquet'))) == 0 and EXT_DRIVE_BACKUP.exists():
        subprocess.run(['rsync', '-ah', f'{EXT_DRIVE_BACKUP}/', f'{DATA_EXT}/'])
else:
    DATA_EXT = cfg.DATA_PROCESSED.parent / 'processed_v2'
    DATA_PROCESSED_LOCAL = cfg.DATA_PROCESSED

def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    # Backwards-compat für legacy NB13-Files (kein hit_bar_offset)
    if 'hit_bar_offset' not in df.columns:
        lp = DATA_PROCESSED_LOCAL / f'labels_{sym}_{tf}_R{R_VALUE}.parquet'
        if lp.exists():
            labels = pd.read_parquet(lp)
            if 'hit_bar_offset' in labels.columns:
                df = df.join(labels[['hit_bar_offset']], how='left')
        if 'hit_bar_offset' not in df.columns:
            df['hit_bar_offset'] = 24
        df['hit_bar_offset'] = df['hit_bar_offset'].fillna(24).astype('int32')
    return df

# Sanity-Check
missing = [s for s in ALL_SYMBOLS if load_ext(s, TF) is None or load_ext(s, TF).empty]
if missing:
    raise SystemExit(f'Missing _extended.parquet für: {missing} — run NB14 Section 2 first.')
print('Alle Extended-Files vorhanden.')

# Train-Pool aufbauen
frames = []
for s in FX_TRAIN_SYMBOLS:
    d = load_ext(s, TF)
    if d is not None and not d.empty:
        d['symbol'] = s
        frames.append(d.astype({c: 'float32' for c in d.select_dtypes(include=['float64']).columns}))
pool = pd.concat(frames, axis=0).sort_index()
del frames
gc.collect()

# Feature-Columns (NB11-Winner-27)
probe = load_ext(FX_TRAIN_SYMBOLS[0], TF)
FEATURE_COLS = [c for c in probe.columns if c not in NON_FEATURE_COLS and c != 'symbol']
print(f'Features: {len(FEATURE_COLS)}')

# Walk-Forward Split
pool_c = pool.dropna(subset=FEATURE_COLS + ['label'])
train_df, val_df, test_df = walk_forward_split(pool_c, TRAIN_END, VAL_END)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
del pool, pool_c
gc.collect()

In [ ]:
# Train identisch zu NB14/NB14b
# core/train/lgbm_trainer.py wurde mit seed=42 + deterministic=True gepatcht (Bug-Fix nach Run 2).
# Damit ist das Modell jetzt reproduzierbar zwischen Runs.
X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = binary_label_for_long(train_df['label']).values
X_val = val_df[FEATURE_COLS].values.astype(np.float32)
y_val = binary_label_for_long(val_df['label']).values

model = train_lgbm(
    pd.DataFrame(X_train, columns=FEATURE_COLS), pd.Series(y_train),
    pd.DataFrame(X_val, columns=FEATURE_COLS), pd.Series(y_val),
)

proba_val = model.predict(X_val)
X_test = test_df[FEATURE_COLS].values.astype(np.float32)
proba_test = model.predict(X_test)

print('=== Probability-Diagnose ===')
print(f'VAL proba range:  [{proba_val.min():.4f}, {proba_val.max():.4f}]  '
      f'mean={proba_val.mean():.4f}  std={proba_val.std():.4f}')
print(f'TEST proba range: [{proba_test.min():.4f}, {proba_test.max():.4f}]  '
      f'mean={proba_test.mean():.4f}  std={proba_test.std():.4f}')
print(f'VAL Quantiles:  p50={np.quantile(proba_val,0.50):.4f}  '
      f'p90={np.quantile(proba_val,0.90):.4f}  '
      f'p97={np.quantile(proba_val,0.97):.4f}  '
      f'p99={np.quantile(proba_val,0.99):.4f}  '
      f'p99.5={np.quantile(proba_val,0.995):.4f}')

# CUTOFF mit FLOOR-SCHUTZ (NB14c Run 2 Debug):
# NB14b zeigte: Probability-Verteilung hat 3 diskrete Bänder
#   < 0.4067 → no signal
#   ~0.4067 Cluster → kein Edge (saturiert)
#   > 0.4096 → Premium-Tier (PF 2.0)
# Wenn dyn-Cutoff ins Cluster fällt (< 0.4096), wird Edge verwässert.
# Floor 0.4070 (knapp über Cluster) schützt davor.
PREMIUM_CUTOFF_HARDCODED_REF = 0.4096   # NB14b-Wert, gelocked durch ANN-012
PREMIUM_CUTOFF_DYNAMIC = float(np.quantile(proba_val, 0.99))
PREMIUM_CUTOFF_FLOOR = 0.4070   # knapp über Cluster, schützt vor Saturation-Bug
PREMIUM_CUTOFF = max(PREMIUM_CUTOFF_DYNAMIC, PREMIUM_CUTOFF_FLOOR)

print(f'\n=== Cutoff-Calibration (mit Floor-Schutz) ===')
print(f'Hardcoded reference (NB14b/ANN-012):  {PREMIUM_CUTOFF_HARDCODED_REF}')
print(f'Dynamic VAL top-1%:                   {PREMIUM_CUTOFF_DYNAMIC:.4f}')
print(f'Floor-Schutz (über 0.4067 Cluster):   {PREMIUM_CUTOFF_FLOOR}')
print(f'Final PREMIUM_CUTOFF:                 {PREMIUM_CUTOFF:.4f}')

if PREMIUM_CUTOFF == PREMIUM_CUTOFF_FLOOR:
    print('  ⚠️  Floor active — dyn-Cutoff war im Cluster-Bereich, Floor angewendet')
elif abs(PREMIUM_CUTOFF - PREMIUM_CUTOFF_HARDCODED_REF) < 0.003:
    print('  ✅ Cutoff matched NB14b-Reference — deterministisches Training funktioniert')
else:
    print(f'  ℹ️  Cutoff weicht ab vom NB14b-Wert um {PREMIUM_CUTOFF - PREMIUM_CUTOFF_HARDCODED_REF:+.4f}')

# Sanity: Cluster-Position prüfen
cluster_low  = float(np.quantile(proba_val, 0.97))
cluster_high = float(np.quantile(proba_val, 0.99))
print(f'\nCluster-Range (p97-p99): [{cluster_low:.4f}, {cluster_high:.4f}]  '
      f'breite={cluster_high-cluster_low:.4f}')

n_premium_test = int((proba_test >= PREMIUM_CUTOFF).sum())
pct_premium = 100 * n_premium_test / len(proba_test)
print(f'TEST bars >= Premium-Cutoff: {n_premium_test} ({pct_premium:.2f}%)')

if n_premium_test == 0:
    print('\n❌ KRITISCH: Cutoff zu hoch — 0 Trades.')
elif n_premium_test < 30:
    print('\n⚠️  Sehr wenige Premium-Trades — Calibration unsicher.')
elif pct_premium > 3:
    print('\n⚠️  >3% Bars klassifiziert — wahrscheinlich Cluster-Saturation, Edge verwässert.')
else:
    print('\n✅ Cutoff-Diagnose okay — fahre mit Filter-Validation fort.')

del X_train, y_train
gc.collect()

## Section 2 — Filter-Definitionen

**HTF-Confirmation:** `htf_ltf_agree_bull` Feature aus dem extended-DataFrame. Diese Feature ist 1 wenn HTF-Trend (1h) und LTF-Bar in die gleiche Richtung zeigen.

**NY-Session:** Hour-Filter UTC 13:00–22:00 (NY-Open bis NY-Close).

In [ ]:
def apply_filters(df, proba, profile_name):
    """
    Returns boolean mask der bars die dieses Profil als Signal akzeptiert.

    profile_name in ('Aggressive', 'Balanced', 'Conservative')
    df muss die feature-cols enthalten ('htf_ltf_agree_bull') + index = timestamp
    """
    cfg = PROFILES[profile_name]
    mask = proba >= PREMIUM_CUTOFF
    if cfg['require_htf']:
        if 'htf_ltf_agree_bull' not in df.columns:
            raise KeyError('htf_ltf_agree_bull fehlt im DataFrame — Feature-Pipeline prüfen')
        mask = mask & (df['htf_ltf_agree_bull'].values == 1)
    if cfg['require_ny']:
        # df index ist UTC timestamp (TradingView-Konvention)
        hour_utc = df.index.hour.values
        ny_mask = (hour_utc >= NY_SESSION_HOUR_START) & (hour_utc < NY_SESSION_HOUR_END)
        mask = mask & ny_mask
    return mask

# Test: Aggressive sollte = pure Premium-Mask sein
test_mask_agg = apply_filters(test_df, proba_test, 'Aggressive')
test_mask_bal = apply_filters(test_df, proba_test, 'Balanced')
test_mask_cons = apply_filters(test_df, proba_test, 'Conservative')
print(f'In-Sample TEST — Filter-Reduktion:')
print(f'  Aggressive   (Premium pur):                 {test_mask_agg.sum():>6} bars')
print(f'  Balanced     (Premium + HTF):               {test_mask_bal.sum():>6} bars')
print(f'  Conservative (Premium + HTF + NY-Session):  {test_mask_cons.sum():>6} bars')

## Section 3 — Per-Profil Metriken auf In-Sample TEST

In [ ]:
def evaluate_filter(df, proba, labels_triple, profile_name, n_symbols, n_bars):
    """
    Vollständige Metriken für ein Profil:
    PF, WR, Trades, Sigs/Tag/Symbol, MDD (per equity curve), avg trade duration.
    """
    mask = apply_filters(df, proba, profile_name)
    n_sig = int(mask.sum())
    if n_sig == 0:
        return {'profile': profile_name, 'n_trades': 0, 'wins': 0, 'losses': 0,
                'pf': 0.0, 'wr': 0.0, 'expR': 0.0, 'sigs_per_day_per_symbol': 0.0,
                'mdd': 0.0, 'avg_duration_bars': 0.0}
    labs = labels_triple[mask.values if hasattr(mask, 'values') else mask]
    wins = int((labs == 1).sum())
    losses = int((labs == -1).sum())
    total = wins + losses + int((labs == 0).sum())
    pf = (wins * R_VALUE) / losses if losses > 0 else (float('inf') if wins > 0 else 0.0)
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    expR = (wins * R_VALUE - losses) / total if total > 0 else 0.0
    sigs_pd = signals_per_day(n_sig, n_bars, TF, n_symbols)

    # MDD aus naiver equity curve (sequentielle Trades, R-based)
    pnl = np.where(labs == 1, R_VALUE, np.where(labs == -1, -1.0, 0.0))
    equity = np.cumsum(pnl)
    running_max = np.maximum.accumulate(equity)
    dd = running_max - equity
    mdd = float(dd.max()) if len(dd) > 0 else 0.0

    # Avg trade duration
    sub = df.iloc[mask.values if hasattr(mask, 'values') else mask]
    if 'hit_bar_offset' in sub.columns and len(sub) > 0:
        avg_dur = float(sub['hit_bar_offset'].mean())
    else:
        avg_dur = float('nan')

    return {
        'profile':                profile_name,
        'n_trades':               n_sig,
        'wins':                   wins,
        'losses':                 losses,
        'pf':                     round(pf, 4),
        'wr':                     round(wr, 4),
        'expR':                   round(expR, 4),
        'sigs_per_day_per_symbol': round(sigs_pd, 4),
        'mdd':                    round(mdd, 4),
        'avg_duration_bars':      round(avg_dur, 2),
    }

test_labels_triple = test_df['label'].values
in_sample_rows = [evaluate_filter(test_df, proba_test, test_labels_triple,
                                  p, len(FX_TRAIN_SYMBOLS), len(test_df))
                  for p in PROFILES.keys()]
in_sample_df = pd.DataFrame(in_sample_rows)
print('=== IN-SAMPLE TEST — per profile ===')
in_sample_df

## Section 4 — Per-Symbol Hold-Out Validation

Wendet die Filter auf jedes der 3 nie trainierten Hold-Out-Symbole (GBPUSD, AUDUSD, USDCHF) an. **Das ist der wichtigste Test** — hier zeigt sich ob die Filter-Mechanik außerhalb der Trainings-Daten hält.

In [ ]:
import pandas as _pd

def filter_holdout_window(df):
    """Restrict to OOS window (>= VAL_END)."""
    val_end_ts = _pd.Timestamp(VAL_END)
    if val_end_ts.tz is None:
        val_end_ts = val_end_ts.tz_localize('UTC')
    return df[df.index >= val_end_ts]

holdout_rows = []
for sym in FX_HOLDOUT_SYMBOLS:
    h = load_ext(sym, TF)
    if h is None or h.empty:
        continue
    h = h.dropna(subset=FEATURE_COLS + ['label'])
    h = filter_holdout_window(h)
    if h.empty:
        continue
    proba_h = model.predict(h[FEATURE_COLS].values.astype(np.float32))
    labels_h = h['label'].values
    for p in PROFILES.keys():
        metrics = evaluate_filter(h, proba_h, labels_h, p, 1, len(h))
        metrics['symbol'] = sym
        holdout_rows.append(metrics)
    del h, proba_h
    gc.collect()

holdout_df = pd.DataFrame(holdout_rows)
print('=== HOLD-OUT (per symbol × profile) ===')
holdout_df

In [ ]:
# Aggregiert über alle 3 Hold-Out-Symbole pro Profil
holdout_agg_rows = []
for p in PROFILES.keys():
    sub = holdout_df[holdout_df['profile'] == p]
    if sub.empty:
        continue
    # PF gewichtet über alle wins/losses
    total_wins   = int(sub['wins'].sum())
    total_losses = int(sub['losses'].sum())
    total_trades = int(sub['n_trades'].sum())
    weighted_pf  = (total_wins * R_VALUE) / total_losses if total_losses > 0 else (float('inf') if total_wins > 0 else 0.0)
    weighted_wr  = total_wins / (total_wins + total_losses) if (total_wins + total_losses) > 0 else 0.0

    # Cross-Symbol Stability CV
    pfs = sub['pf'].replace([np.inf, -np.inf], np.nan).dropna()
    cv = float(pfs.std() / pfs.mean()) if len(pfs) >= 2 and pfs.mean() != 0 else float('nan')

    holdout_agg_rows.append({
        'profile':                p,
        'symbols':                len(sub),
        'total_trades':           total_trades,
        'wins':                   total_wins,
        'losses':                 total_losses,
        'pf_weighted':            round(weighted_pf, 4),
        'wr_weighted':            round(weighted_wr, 4),
        'pf_per_symbol_min':      round(pfs.min(), 4) if len(pfs) > 0 else None,
        'pf_per_symbol_max':      round(pfs.max(), 4) if len(pfs) > 0 else None,
        'cross_symbol_cv':        round(cv, 4) if not np.isnan(cv) else None,
        'avg_sigs_per_day_sym':   round(sub['sigs_per_day_per_symbol'].mean(), 4),
        'max_mdd':                round(sub['mdd'].max(), 4),
        'avg_duration_bars':      round(sub['avg_duration_bars'].mean(), 2),
    })

holdout_agg_df = pd.DataFrame(holdout_agg_rows)
print('=== HOLD-OUT AGGREGATED (per profile, weighted across 3 symbols) ===')
holdout_agg_df

## Section 5 — Quality-Anchor Check per Profile (ANN-010)

In [ ]:
quality_rows = []
for p in PROFILES.keys():
    is_row  = next((r for r in in_sample_rows if r['profile'] == p), None)
    ho_row  = next((r for r in holdout_agg_rows if r['profile'] == p), None)
    if is_row is None or ho_row is None:
        continue

    # Bauen wir ein metrics-dict im Format das check_quality_anchor erwartet
    metrics_for_check = {
        'premium_pf_oos':         is_row['pf'],
        'premium_pf_holdout':     ho_row['pf_weighted'],
        'min_pf_per_symbol':      ho_row['pf_per_symbol_min'],
        'stability_cv':           ho_row['cross_symbol_cv'],
        'min_pf_per_year':        None,  # NB14 hat Yearly-Daten, hier nicht neu berechnet
        'min_trades_per_year_tier': None,
        'mdd':                    ho_row['max_mdd'],
    }
    try:
        passed, severity, details = check_quality_anchor(metrics_for_check)
    except Exception as e:
        passed, severity, details = (False, f'check_error: {e}', {})

    quality_rows.append({
        'profile':       p,
        'in_sample_pf':  is_row['pf'],
        'holdout_pf':    ho_row['pf_weighted'],
        'min_pf_symbol': ho_row['pf_per_symbol_min'],
        'mdd':           ho_row['max_mdd'],
        'sigs_per_day':  ho_row['avg_sigs_per_day_sym'],
        'quality_passed': passed,
        'severity':       severity,
    })

quality_df = pd.DataFrame(quality_rows)
print('=== QUALITY-ANCHOR per profile ===')
quality_df

## Section 6 — Final Verdict + Profile-Calibration

In [ ]:
verdict = {
    'experiment_id':   EXPERIMENT_ID,
    'run_date':        RUN_DATE,
    'git_commit':      GIT_COMMIT,
    'premium_cutoff':  PREMIUM_CUTOFF,
    'profiles':        {},
}

for p in PROFILES.keys():
    is_row = next((r for r in in_sample_rows if r['profile'] == p), {})
    ho_row = next((r for r in holdout_agg_rows if r['profile'] == p), {})
    q_row  = next((r for r in quality_rows if r['profile'] == p), {})

    verdict['profiles'][p] = {
        'filter_stack': PROFILES[p],
        'in_sample': {
            'pf':                   is_row.get('pf'),
            'wr':                   is_row.get('wr'),
            'sigs_per_day_symbol':  is_row.get('sigs_per_day_per_symbol'),
            'mdd':                  is_row.get('mdd'),
            'avg_duration':         is_row.get('avg_duration_bars'),
        },
        'holdout': {
            'pf_weighted':          ho_row.get('pf_weighted'),
            'wr_weighted':          ho_row.get('wr_weighted'),
            'min_pf_per_symbol':    ho_row.get('pf_per_symbol_min'),
            'max_pf_per_symbol':    ho_row.get('pf_per_symbol_max'),
            'cross_symbol_cv':      ho_row.get('cross_symbol_cv'),
            'avg_sigs_per_day':     ho_row.get('avg_sigs_per_day_sym'),
            'max_mdd':              ho_row.get('max_mdd'),
        },
        'quality_anchor': {
            'passed':   q_row.get('quality_passed'),
            'severity': q_row.get('severity'),
        },
    }

# Decision: locke alle 3 Profile wenn alle Quality-Anchor passen
all_passed = all(q_row.get('quality_passed', False) for q_row in quality_rows)
verdict['decision'] = {
    'all_profiles_passed_quality_anchor': all_passed,
    'v1_ready':  all_passed,
    'next_step': 'NB15 Pine-Router-V1-Validation' if all_passed
                 else 'Profile mit FAIL-Status re-evaluieren oder Filter anpassen',
}

print('=' * 70)
print(f'NB14c VERDICT — Run {RUN_DATE}')
print('=' * 70)
for p, data in verdict['profiles'].items():
    ho = data['holdout']
    qa = data['quality_anchor']
    print(f'{p:14s}  PF={ho["pf_weighted"]:.2f}  WR={ho["wr_weighted"]:.2%}  '
          f'Sigs={ho["avg_sigs_per_day"]:.2f}/Tag/Sym  MDD={ho["max_mdd"]:.1f}R  '
          f'Quality={qa["severity"]}')
print('-' * 70)
print(f'V1-Ready: {verdict["decision"]["v1_ready"]}')
print(f'Next Step: {verdict["decision"]["next_step"]}')
print('=' * 70)

## Section 7 — Persistence (`/results/nb14c/`)

In [ ]:
# Persist CSVs
in_sample_df.to_csv(OUTPUT_DIR / 'metrics' / f'in_sample_per_profile_{RUN_DATE}.csv', index=False)
holdout_df.to_csv(OUTPUT_DIR / 'metrics' / f'holdout_per_symbol_profile_{RUN_DATE}.csv', index=False)
holdout_agg_df.to_csv(OUTPUT_DIR / 'metrics' / f'holdout_aggregated_per_profile_{RUN_DATE}.csv', index=False)
quality_df.to_csv(OUTPUT_DIR / 'metrics' / f'quality_anchor_per_profile_{RUN_DATE}.csv', index=False)

# Full JSON snapshot
snapshot_path = OUTPUT_DIR / 'summaries' / f'nb14c_full_snapshot_{RUN_DATE}.json'
with open(snapshot_path, 'w') as f:
    json.dump({
        'verdict':        verdict,
        'in_sample':      in_sample_df.replace([np.inf, -np.inf], 'inf').to_dict(orient='records'),
        'holdout_per_sym': holdout_df.replace([np.inf, -np.inf], 'inf').to_dict(orient='records'),
        'holdout_agg':    holdout_agg_df.replace([np.inf, -np.inf], 'inf').to_dict(orient='records'),
        'quality':        quality_df.to_dict(orient='records'),
    }, f, indent=2, default=str)
print(f'Snapshot → {snapshot_path}')
print('CSVs gespeichert in', OUTPUT_DIR / 'metrics')

## Section 8 — Auto-Push to GitHub

Setup: `GITHUB_TOKEN` in Colab Secrets (siehe `/docs/colab_auto_push.md`).

In [ ]:
import shutil
if not IN_COLAB:
    print('Local run — skip push.')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception as e:
        GH_TOKEN = None
        print(f'ERROR cannot read GITHUB_TOKEN: {e}')

    if GH_TOKEN:
        NB_TAG = 'nb14c'
        TMP_REPO = Path('/tmp/pace-algo-push')
        if TMP_REPO.exists():
            shutil.rmtree(TMP_REPO)
        subprocess.run(['git', 'clone', '--quiet',
                        f'https://{GH_TOKEN}@github.com/ecoNC/pace-algo.git', str(TMP_REPO)], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.name', 'ecoNC'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.email',
                        'ecoNC@users.noreply.github.com'], check=True)

        # FIX (vs Run 1): pull --rebase VOR dem Copy.
        # Run 1 ist gescheitert weil die kopierten Files existierende Files (vom 1. Run)
        # ueberschrieben → unstaged changes → 'git pull --rebase' bricht ab.
        # Loesung: pull zuerst (clean working tree direkt nach clone), DANN copy.
        # --autostash zusaetzlich fuer Robustness wenn jemand parallel pusht.
        subprocess.run(['git', '-C', str(TMP_REPO), 'pull', '--rebase', '--autostash', '--quiet',
                        'origin', 'main'], check=True)
        print('Pulled latest from origin/main (clean rebase)')

        # JETZT erst die neuen Result-Files in das frisch gepulle Repo kopieren.
        copied = []
        for f in sorted(OUTPUT_DIR.rglob(f'*{RUN_DATE}*')):
            if not f.is_file(): continue
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        # Config snapshot mit dazu
        for f in sorted((OUTPUT_DIR / 'config_snapshots').glob(f'*{EXPERIMENT_ID}*')):
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        print(f'Copied {len(copied)} files')

        subprocess.run(['git', '-C', str(TMP_REPO), 'add', 'results/'], check=True)
        r_status = subprocess.run(['git', '-C', str(TMP_REPO), 'status', '--porcelain'],
                                    capture_output=True, text=True)
        if not r_status.stdout.strip():
            print('Nothing to commit (results already on origin).')
        else:
            msg = f'{NB_TAG.upper()} secondary-filter validation {RUN_DATE} ({len(copied)} files)'
            subprocess.run(['git', '-C', str(TMP_REPO), 'commit', '-m', msg], check=True)
            sha = subprocess.run(['git', '-C', str(TMP_REPO), 'rev-parse', '--short', 'HEAD'],
                                   capture_output=True, text=True).stdout.strip()
            subprocess.run(['git', '-C', str(TMP_REPO), 'push', 'origin', 'main'], check=True)
            print(f'✓ Pushed as ecoNC ({sha})')
            print(f'  https://github.com/ecoNC/pace-algo/commit/{sha}')
        shutil.rmtree(TMP_REPO)

## Section 9 — Final Verdict-Notes (für Claude post-Run)

Nach dem Run analysiert Claude die JSON und:
1. Locked die finalen Sigs/Tag-Zahlen in **ANN-012** (Section 4 Profile-Mapping)
2. Updated `research/timeframe_comparisons.md` mit den Hold-Out-Validation-Daten
3. Updated `docs/model_registry.md` mit FX-Modell Final-Calibration
4. Wenn ein Profil FAILED: Filter anpassen oder Profil entfernen (Quality-Anchor strict)
5. Falls alles passt: Phase D (NB15 Pine-Router-V1-Validation) starten

Erwartete Ergebnis-Range (basierend auf ANN-012-Planung):
- Aggressive: PF ~2.0+, ~3.5 Sigs/Tag/Symbol
- Balanced: PF ~2.2+, ~3.0 Sigs/Tag/Symbol
- Conservative: PF ~2.5+, ~1.5 Sigs/Tag/Symbol

**Wenn Conservative deutlich unter ~1 Sigs/Tag fällt:** NY-Session-Filter zu strikt, Range anpassen oder Conservative droppen → V1 = nur Aggressive+Balanced.